In [40]:
import yfinance as yf
from scipy.optimize import minimize
import numpy as np
import datetime
import pandas as pd

today = datetime.date.today()
print(f"Today's date: {today}")

Today's date: 2026-05-01


# Get the last price of a stock

In [41]:
def get_last_price(ticker):
    data = ticker.history(period="1d")
    price = data['Close'].iloc[0]
    official_name = ticker.info['longName']
    return f"The last price of {official_name} ({ticker.ticker}) is ${price:.2f}"

In [73]:
get_last_price(yf.Ticker("AAPL"))

'The last price of Apple Inc. (AAPL) is $280.14'

In [71]:
def get_price_on_date(ticker, date=None):
    t = yf.Ticker(ticker)
    
    use_default_date = date is None
    
    if date is None:
        date = datetime.date.today()
    else:
        date = datetime.datetime.strptime(date, "%Y-%m-%d").date()
    
    data = pd.DataFrame(t.history(start=date - datetime.timedelta(days=5), end=date + datetime.timedelta(days=5))['Close'])    
    if data.empty:
        return f"No price data available for {t.ticker} around {date}."
    
    data['Date'] = data.index.date
    data['DateDiff'] = np.abs(data['Date'] - date)
    nearest_row = data.loc[data['DateDiff'].idxmin()]
    price = nearest_row['Close']
    actual_date = nearest_row['Date']
    official_name = t.info['longName']
    
    if use_default_date:
        return f"The last price of {official_name} ({t.ticker}) is ${price:.2f} as of {actual_date}."
    else:
        return f"The price of {official_name} ({t.ticker}) nearest to {date} was ${price:.2f} on {actual_date}."

In [80]:
get_price_on_date("F", "2024-01-15")

'The price of Ford Motor Company (F) nearest to 2024-01-15 was $9.86 on 2024-01-16.'

# Get the company's profile

In [5]:
def get_company_profile(ticker):
    info = ticker.info
    official_name = info['longName']
    sector = info.get('sector', 'N/A')
    industry = info.get('industry', 'N/A')
    description = info.get('longBusinessSummary', 'No description available.')
    return (
        f"{official_name} operates in the {sector} sector and {industry} industry. "
        f"Company profile: {description}"
    )

In [6]:
(get_company_profile(yf.Ticker("F")))

'Ford Motor Company operates in the Consumer Cyclical sector and Auto Manufacturers industry. Company profile: Ford Motor Company develops, delivers, and services Ford trucks, sport utility vehicles, commercial vans and cars, and Lincoln luxury vehicles in the United States, Canada, the United Kingdom, Mexico, and internationally. It operates through Ford Blue, Ford Model e, Ford Pro, and Ford Credit segments. The company sells Ford and Lincoln internal combustion engine and hybrid vehicles, electric vehicles, service parts, accessories, and digital services for retail customers; develops EV and digital vehicle technologies, and software; and provides telematics and EV charging solutions. It also sells Ford and Lincoln vehicles, service parts, and accessories through distributors and dealers, as well as through dealerships to commercial fleet customers, daily rental car companies, and governments. In addition, it engages in vehicle-related financing and leasing activities to and throug

# Forex

In [7]:
def get_usd_mxn():
    price = yf.Ticker("MXN=X").history(period="1d")['Close'].iloc[0]
    return f"The current exchange rate for USD/MXN is {price:.2f} MXN per USD."

def get_mxn_usd():
    price = yf.Ticker("MXN=X").history(period="1d")['Close'].iloc[0]
    return f"The current exchange rate for MXN/USD is {1/price:.4f} USD per MXN."

def get_eur_mxn():
    price = yf.Ticker("EURMXN=X").history(period="1d")['Close'].iloc[0]
    return f"The current exchange rate for EUR/MXN is {price:.2f} MXN per EUR."

def get_mxn_eur():
    price = yf.Ticker("EURMXN=X").history(period="1d")['Close'].iloc[0]
    return f"The current exchange rate for MXN/EUR is {1/price:.4f} EUR per MXN."

In [8]:
get_usd_mxn(), get_mxn_usd(), get_eur_mxn(), get_mxn_eur()

('The current exchange rate for USD/MXN is 17.52 MXN per USD.',
 'The current exchange rate for MXN/USD is 0.0571 USD per MXN.',
 'The current exchange rate for EUR/MXN is 20.47 MXN per EUR.',
 'The current exchange rate for MXN/EUR is 0.0489 EUR per MXN.')

# Portfolio Optimization

In [14]:
def min_variance_portfolio(tickers):
    data = yf.download(tickers, period="1y", progress=False)['Close'][tickers]
    returns = data.pct_change().dropna()
    cov_matrix = returns.cov()
    mean_rt = returns.mean()

    variance = lambda w: w.T @ cov_matrix @ w
    x0 = np.ones(len(tickers)) / len(tickers)
    bounds = [(0.05, 3)] * len(tickers)
    constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    result = minimize(variance, x0, bounds=bounds, constraints=constraints, tol=1e-16, method='SLSQP')
    return (
        f"Optimal weights for minimum variance portfolio: {dict(zip(tickers, result.x.round(4)))}\n"
        f"Expected annual return: {(mean_rt @ result.x * 252):.2%}\n"
        f"Annualized volatility: {(np.sqrt(result.fun) * np.sqrt(252)):.2%}"
    )

In [15]:
min_variance_portfolio(["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"])

"Optimal weights for minimum variance portfolio: {'AAPL': np.float64(0.3745), 'MSFT': np.float64(0.3659), 'GOOGL': np.float64(0.1596), 'AMZN': np.float64(0.05), 'TSLA': np.float64(0.05)}\nExpected annual return: 31.69%\nAnnualized volatility: 17.50%"

In [115]:
import os
from dotenv import load_dotenv
import requests
import datetime

load_dotenv()

BANXICO_TOKEN = os.getenv("BANXICO_TOKEN")

def get_cetes_28(date: str | None = None) -> str:
    URL = "https://www.banxico.org.mx/SieAPIRest/service/v1/series/SF46405/datos"
    headers = {
        "Bmx-Token": BANXICO_TOKEN,
        "Content-Type": "application/json",
    }
    try:
        response = requests.get(URL, headers=headers)
        response.raise_for_status()

        obs_list = response.json()["bmx"]["series"][0]["datos"]

        if date is None:
            obs = obs_list[-1]
        else:
            target = datetime.datetime.strptime(date, "%Y-%m-%d")
            obs = min(
                obs_list,
                key=lambda o: abs(datetime.datetime.strptime(o["fecha"], "%d/%m/%Y") - target),
            )

        fecha = obs["fecha"]
        fecha = datetime.datetime.strptime(fecha, "%d/%m/%Y").strftime("%Y-%m-%d")
        value = float(obs["dato"])
        label = f"nearest to {date}" if date else "most recent"
        return f"The CETES 28-day rate ({label}) is {value:.4f}% as of {fecha}."

    except ValueError:
        return f"Invalid date format '{date}'. Please use YYYY-MM-DD (e.g. 2024-01-15)."
    except Exception as exc:
        return f"Error fetching CETES 28-day rate: {exc}"

In [132]:
def get_exchange_rate(base: str, quote: str, date: str | None = None) -> str:
    base = base.strip().upper()
    quote = quote.strip().upper()
    ticker_symbol = f"{base}{quote}=X"
 
    try:
        if date is None:
            target_date = datetime.date.today()
        else:
            target_date = datetime.datetime.strptime(date, "%Y-%m-%d").date()
 
        t = yf.Ticker(ticker_symbol)
        data = t.history(
            start=target_date - datetime.timedelta(days=7),
            end=target_date + datetime.timedelta(days=7),
        )
 
        if data.empty:
            return (
                f"No exchange rate data found for {base}/{quote} ({ticker_symbol}). "
                f"Verify that both currency codes are valid ISO 4217 codes."
            )
 
        data["Date"] = data.index.date
        data["DateDiff"] = data["Date"].apply(lambda d: abs((d - target_date).days))
        nearest = data.loc[data["DateDiff"].idxmin()]
        rate = nearest["Close"]
        actual_date = nearest["Date"]
        date_label = f"nearest to {date}" if date else "most recent"
        return f"The exchange rate for {base}/{quote} ({date_label}) is {rate:.6f} as of {actual_date}." 
        
    except ValueError:
        return f"Invalid date format '{date}'. Please use YYYY-MM-DD (e.g. 2024-01-15)."
    except Exception as exc:
        return f"Error fetching exchange rate for {base}/{quote}: {exc}"

In [135]:
get_exchange_rate("JPY", "EUR", "2010-09-01")

'The exchange rate for JPY/EUR (nearest to 2010-09-01) is 0.009370 as of 2010-09-01.'

In [137]:
def get_udis(date: str | None = None) -> str:
    URL = "https://www.banxico.org.mx/SieAPIRest/service/v1/series/SP68257/datos"
    headers = {
        "Bmx-Token": BANXICO_TOKEN,
        "Content-Type": "application/json",
    }
    try:
        response = requests.get(URL, headers=headers)
        response.raise_for_status()

        obs_list = response.json()["bmx"]["series"][0]["datos"]

        if date is None:
            obs = obs_list[-1]
        else:
            target = datetime.datetime.strptime(date, "%Y-%m-%d")
            obs = min(
                obs_list,
                key=lambda o: abs(datetime.datetime.strptime(o["fecha"], "%d/%m/%Y") - target),
            )

        fecha = obs["fecha"]
        fecha = datetime.datetime.strptime(fecha, "%d/%m/%Y").strftime("%Y-%m-%d")
        value = float(obs["dato"])
        label = f"nearest to {date}" if date else "most recent"
        return f"The value of UDIs in Mexico ({label}) is {value:.4f} MXN as of {fecha}."

    except ValueError:
        return f"Invalid date format '{date}'. Please use YYYY-MM-DD (e.g. 2024-01-15)."
    except Exception as exc:
        return f"Error fetching UDIs value in Mexico: {exc}"

In [138]:
get_udis()

'The value of UDIs in Mexico (most recent) is 8.8410 MXN as of 2026-05-10.'